# Part 3: UI build (12 min)

Replace the template's hello screen with the Specter Mutual claims portal: a claim form and the decision on the left, the adjuster console on the right. Each cell writes one file into `apps/specter-mutual/src/`. Then you open the App Builder and watch it run in the Preview.

The files come in four groups:

| Group | Files | What it does |
| --- | --- | --- |
| Style system | `theme.ts`, `useSpecterTheme.ts` | The palette, fonts and shared styles. Every component takes its look from here. |
| Logic | `policy.ts`, `claims.ts`, `apiKey.ts` | The policy engine (the same rules as `policy.py`), the pipeline call, and the Anthropic key check |
| Components | `TopNav` … `ApiKeyModal` | The pieces of the page |
| The page | `App.tsx`, `icon.svg`, `README.md` | Puts it together, plus the app's icon and store listing |

`claims.ts` is the part that talks to RocketRide: it starts the `claims.pipe` pipeline from Part 2 with `useExisting`, sends each claim as `text/plain`, and runs the agent's answer through `policy.ts` before anyone sees it.

In [ ]:
# Check Parts 1 and 2 left what this part needs, and make the components folder
%run ./specter-mutual.ipynb
missing = [p for p in (APP_ROOT / "package.json", PIPELINES / "claims.pipe") if not p.exists()]
if missing:
    raise RuntimeError("Missing " + ", ".join(p.relative_to(WORKSPACE).as_posix() for p in missing)
                       + ". Run the guide's Start the project step, then Part 1 and Part 2 (or their skip-ahead cells) first.")
(APP_ROOT / "src" / "components").mkdir(exist_ok=True)
print("Ready to write the portal into", APP_ROOT.relative_to(WORKSPACE).as_posix())

## Style system

The Specter Mutual look: light only, ink bars, monospace labels and solid decision badges. The palette lives in `--sm-*` CSS variables so the portal looks the same whatever theme the RocketRide shell is in.

In [ ]:
%%writefile apps/specter-mutual/src/theme.ts
/**
 * Specter Mutual palette — the Zero to Agent codelab look (ink bars, mono
 * uppercase labels, solid badges). Light only, by choice: it replaces the
 * shell's --rr-* colours with app-scoped --sm-* variables, so the portal
 * looks the same whatever theme the shell is in.
 */

import type React from 'react';
import type { Decision } from './policy';

// =============================================================================
// PALETTE
// =============================================================================

export const PALETTE = {
	'--sm-ink': '#0B0A1F', // strongest text; selected segments
	'--sm-ink-2': '#272552', // body text
	'--sm-mid': '#55527C', // labels
	'--sm-dim': '#8785A6', // muted text
	'--sm-line': '#E6E5F0',
	'--sm-line-2': '#CFCDE2',
	'--sm-paper': '#FFFFFF',
	'--sm-tint': '#F7F7FC',
	'--sm-tint-2': '#EFEEF7',
	'--sm-bar': '#0B0A1F', // panel header bars
	'--sm-bar-text': '#FFFFFF',
	'--sm-bar-sub': '#9A97BE',
	'--sm-accent': '#FF3B20',
	'--sm-accent-d': '#D32B12',
	'--sm-accent-soft': '#FFF0ED',
	'--sm-ok': '#0C7D56',
	'--sm-ok-soft': '#E9F6F1',
	'--sm-warn': '#A96605',
	'--sm-warn-soft': '#FCF3E4',
	'--sm-bad': '#C32B18',
	'--sm-bad-soft': '#FCEDEA',
	'--sm-on-solid': '#FFFFFF', // text on solid badges
};

// =============================================================================
// SHARED STYLES
// =============================================================================

export const fonts = {
	sans: '"IBM Plex Sans", -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif',
	mono: '"IBM Plex Mono", ui-monospace, SFMono-Regular, Menlo, Consolas, monospace',
};

/** Mono uppercase field label, as in the codelab simulator. */
export const label: React.CSSProperties = {
	display: 'block',
	fontFamily: fonts.mono,
	fontSize: 10,
	letterSpacing: '0.11em',
	textTransform: 'uppercase',
	fontWeight: 600,
	color: 'var(--sm-mid)',
	marginBottom: 6,
};

export const input: React.CSSProperties = {
	width: '100%',
	boxSizing: 'border-box',
	fontFamily: fonts.sans,
	fontSize: 14,
	color: 'var(--sm-ink)',
	background: 'var(--sm-paper)',
	border: '1px solid var(--sm-line-2)',
	borderRadius: 6,
	padding: '8px 10px',
};

const buttonBase: React.CSSProperties = {
	fontFamily: fonts.sans,
	fontSize: 14,
	fontWeight: 600,
	borderRadius: 6,
	padding: '9px 16px',
	cursor: 'pointer',
	transition: 'opacity .14s',
};

export const buttons: Record<'primary' | 'secondary', React.CSSProperties> = {
	primary: { ...buttonBase, background: 'var(--sm-ink)', color: 'var(--sm-paper)', border: '1px solid var(--sm-ink)' },
	secondary: { ...buttonBase, background: 'var(--sm-paper)', color: 'var(--sm-ink)', border: '1px solid var(--sm-line-2)' },
};

const DECISION_COLOR: Record<Decision, string> = { approve: 'var(--sm-ok)', deny: 'var(--sm-bad)', escalate: 'var(--sm-warn)' };

/** Solid uppercase verdict badge; `small` for table rows. */
export function badge(decision: Decision, small = false): React.CSSProperties {
	return {
		display: 'inline-block',
		fontFamily: fonts.mono,
		fontSize: small ? 11 : 13,
		fontWeight: 600,
		letterSpacing: '0.06em',
		textTransform: 'uppercase',
		padding: small ? '3px 8px' : '6px 14px',
		borderRadius: small ? 4 : 5,
		color: 'var(--sm-on-solid)',
		background: DECISION_COLOR[decision],
	};
}

/** The "Policy override" pill. */
export const overridePill: React.CSSProperties = {
	display: 'inline-block',
	fontFamily: fonts.mono,
	fontSize: 10.5,
	fontWeight: 600,
	letterSpacing: '0.1em',
	textTransform: 'uppercase',
	color: 'var(--sm-accent-d)',
	background: 'var(--sm-accent-soft)',
	padding: '4px 9px',
	borderRadius: 4,
};

/** Tinted note block with a coloured left rule. */
export function note(tone: 'neutral' | 'bad' = 'neutral'): React.CSSProperties {
	return {
		fontSize: 14,
		lineHeight: 1.55,
		color: 'var(--sm-ink-2)',
		margin: 0,
		padding: '12px 14px',
		borderRadius: 6,
		background: tone === 'bad' ? 'var(--sm-bad-soft)' : 'var(--sm-tint)',
		borderLeft: `3px solid ${tone === 'bad' ? 'var(--sm-bad)' : 'var(--sm-line-2)'}`,
	};
}

In [ ]:
%%writefile apps/specter-mutual/src/useSpecterTheme.ts
/**
 * Root style for the Specter palette: the --sm-* variables plus base font
 * and colours. Also loads IBM Plex and the focus ring once.
 */

import type React from 'react';
import { useEffect } from 'react';
import { fonts, PALETTE } from './theme';

const FONT_HREF = 'https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500;600&family=IBM+Plex+Sans:wght@400;500;600;700&display=swap';

/** Marks the portal's root element; the focus rule below is scoped to it. */
export const ROOT_ATTR = 'data-specter-root';

/**
 * The codelab's focus ring: a red outline offset from the field. Inline
 * styles cannot express :focus-visible, so this one rule is injected.
 */
const FOCUS_CSS = `[${ROOT_ATTR}] :focus-visible { outline: 2px solid var(--sm-accent); outline-offset: 3px; border-radius: 3px; }
[${ROOT_ATTR}] input:focus-visible, [${ROOT_ATTR}] textarea:focus-visible { border-radius: 6px; }`;

const ROOT_STYLE: React.CSSProperties = {
	...(PALETTE as React.CSSProperties),
	fontFamily: fonts.sans,
	color: 'var(--sm-ink-2)',
	background: 'var(--sm-paper)',
};

/** Load IBM Plex and the focus ring once. If the font is blocked, the stacks fall back to system fonts. */
function ensureAssets(): void {
	if (!document.querySelector(`link[href="${FONT_HREF}"]`)) {
		const link = document.createElement('link');
		link.rel = 'stylesheet';
		link.href = FONT_HREF;
		document.head.appendChild(link);
	}
	if (!document.getElementById('specter-focus')) {
		const style = document.createElement('style');
		style.id = 'specter-focus';
		style.textContent = FOCUS_CSS;
		document.head.appendChild(style);
	}
}

export function useSpecterTheme(): React.CSSProperties {
	useEffect(ensureAssets, []);
	return ROOT_STYLE;
}

## Logic

`policy.ts` is `policy.py` in TypeScript, line for line. `claims.ts` calls the pipeline: one task per session, attached with `useExisting`, the claim sent as `text/plain`, and an `LLM error` answer turned into a real error. `apiKey.ts` checks for your Anthropic key in RocketRide Variables, and saves it there if you paste one.

In [ ]:
%%writefile apps/specter-mutual/src/policy.ts
/**
 * Policy engine — a 1:1 port of policy.py at the workspace root. Same rule
 * order, same output fields, same wording. Both are checked against
 * policy-cases.json so they cannot drift apart.
 */

// =============================================================================
// TYPES
// =============================================================================

export type Decision = 'approve' | 'deny' | 'escalate';

export interface Claim {
	claimId: string;
	policyholder: string;
	tier: string;
	incidentDate: string;
	filedDate: string;
	amount: number;
	description: string;
}

export interface AgentDecision {
	category?: string;
	recommendation?: string;
	reasoning?: string;
	letter?: string;
	[key: string]: unknown;
}

export interface Adjudication {
	claimId: string;
	category: string;
	decision: Decision;
	payout: number;
	letter: string;
	reasoning: string;
	policyNote: string | null;
}

// =============================================================================
// POLICY
// =============================================================================

/** Coverage limits in dollars. null = not covered. */
export const POLICY: Record<string, Record<string, number | null>> = {
	poltergeist_damage: { basic: 1000, premium: 5000 },
	cold_spots_hvac: { basic: null, premium: 2000 },
	ectoplasm_cleanup: { basic: 500, premium: 1500 },
	possession: { basic: null, premium: null },
};
export const FILING_WINDOW_DAYS = 30;
export const HUMAN_REVIEW_OVER = 3000;
const DECISIONS: ReadonlySet<string> = new Set(['approve', 'deny', 'escalate']);

const UNREADABLE: AgentDecision = {
	category: 'other',
	recommendation: 'escalate',
	reasoning: '(agent output unreadable)',
	letter: '',
};

// =============================================================================
// HELPERS
// =============================================================================

/** Python's f"{n:,}" for whole dollars. */
const fmtInt = (n: number): string => n.toLocaleString('en-US', { maximumFractionDigits: 0 });

/** Python's f"{n:,.2f}". */
const fmtMoney = (n: number): string => n.toLocaleString('en-US', { minimumFractionDigits: 2, maximumFractionDigits: 2 });

const MS_PER_DAY = 86_400_000;

/** Whole days between two YYYY-MM-DD dates, like Python's date subtraction. */
const daysBetween = (from: string, to: string): number => Math.round((Date.parse(`${to}T00:00:00Z`) - Date.parse(`${from}T00:00:00Z`)) / MS_PER_DAY);

/** The agent is told to return JSON, but agents don't always listen. */
export function parseDecision(raw: unknown): AgentDecision {
	if (raw !== null && typeof raw === 'object' && !Array.isArray(raw)) return raw as AgentDecision;
	if (typeof raw !== 'string') return { ...UNREADABLE };
	const start = raw.indexOf('{');
	const end = raw.lastIndexOf('}') + 1;
	if (start < 0 || end <= 0) return { ...UNREADABLE };
	try {
		return JSON.parse(raw.slice(start, end)) as AgentDecision;
	} catch {
		return { ...UNREADABLE };
	}
}

// =============================================================================
// ENGINE
// =============================================================================

export function adjudicate(claim: Claim, rawDecision: unknown): Adjudication {
	const agent = parseDecision(rawDecision);
	const category = agent.category ?? 'other';
	let recommended = agent.recommendation as string;
	if (!DECISIONS.has(recommended)) recommended = 'escalate';

	const amount = Number(claim.amount);
	const tier = String(claim.tier).toLowerCase();
	const daysLate = daysBetween(claim.incidentDate, claim.filedDate);

	// 1. Hard rules. These ignore what the agent recommended.
	let limit: number | null = null;
	let rule: string | null = null;
	let decision: Decision;
	if (daysLate > FILING_WINDOW_DAYS) {
		decision = 'deny';
		rule = `Filed ${daysLate} days after the incident (limit is ${FILING_WINDOW_DAYS}).`;
	} else if (!(category in POLICY)) {
		decision = 'escalate';
		rule = `Category '${category}' is not in the policy.`;
	} else if (POLICY[category][tier] === null) {
		decision = 'deny';
		rule = `'${category}' is not covered on the ${tier} tier.`;
	} else if (amount > HUMAN_REVIEW_OVER) {
		decision = 'escalate';
		rule = `Claims over $${fmtInt(HUMAN_REVIEW_OVER)} require a human adjuster.`;
	} else {
		// 2. Within policy: the agent may be stricter, never more generous.
		limit = POLICY[category][tier];
		decision = recommended as Decision;
	}

	// 3. Payout never exceeds the coverage limit or the amount claimed.
	const payout = decision === 'approve' ? Math.min(amount, limit as number) : 0;

	const notes: string[] = [];
	if (decision !== recommended) notes.push(`Policy override: agent recommended ${recommended}. ${rule}`);
	if (decision === 'approve' && payout < amount) {
		rule = `Payout capped at the $${fmtInt(limit as number)} ${tier} coverage limit.`;
		notes.push(rule);
	}

	// 4. If the rules changed anything, the customer gets a compliant template
	//    instead of the agent's letter, which was written before the rules ran.
	let letter = String(agent.letter ?? '');
	if (notes.length > 0 || !letter) {
		const outcome = { approve: `approved for $${fmtMoney(payout)}`, deny: 'denied', escalate: 'referred to a human adjuster' }[decision];
		const reason = rule ? ` Reason: ${rule}` : '';
		letter = `Dear ${claim.policyholder},\n\nYour claim ${claim.claimId} has been ${outcome}.${reason}\n\nSpecter Mutual Claims`;
	}

	return {
		claimId: claim.claimId,
		category,
		decision,
		payout,
		letter,
		reasoning: String(agent.reasoning ?? ''),
		policyNote: notes.length > 0 ? notes.join(' ') : null,
	};
}

In [ ]:
%%writefile apps/specter-mutual/src/claims.ts
/**
 * Claim plumbing — builds claims from the form, runs them through the
 * claims pipeline, and adjudicates the agent's answer. Nothing the agent
 * says leaves this module without passing through adjudicate().
 */

import type { useShellConnection } from 'shell';
import claimsPipe from '../../../pipelines/claims.pipe';
import { adjudicate, type Adjudication, type Claim } from './policy';

// =============================================================================
// TYPES
// =============================================================================

export interface ClaimForm {
	policyholder: string;
	tier: 'basic' | 'premium';
	incidentDate: string;
	amount: string;
	description: string;
}

export interface ClaimResult extends Adjudication {
	claim: Claim;
}

/** The shell's client — its bundled type, not the standalone rocketride package's. */
export type ShellClient = NonNullable<ReturnType<typeof useShellConnection>['client']>;

type PipelineConfig = NonNullable<Parameters<ShellClient['use']>[0]>['pipeline'];

// =============================================================================
// PURE HELPERS
// =============================================================================

const FIRST_CLAIM_NUMBER = 1001;

/**
 * The pipeline reports model failures as an ordinary answer. The LLM node
 * writes "**LLM error** — …", the agent "LLM error: Exception: …".
 */
const LLM_ERROR = /^\s*(\*\*)?LLM error\b/;

/** Local date as YYYY-MM-DD. */
export function todayIso(now: Date = new Date()): string {
	const pad = (n: number) => String(n).padStart(2, '0');
	return `${now.getFullYear()}-${pad(now.getMonth() + 1)}-${pad(now.getDate())}`;
}

const MAX_WHOLE_DIGITS = 9;

/**
 * Filter typed or pasted text down to a dollar amount: digits and one
 * decimal point with at most two places. Anything else is dropped.
 */
export function sanitizeAmount(raw: string): string {
	const [whole, ...rest] = raw.replace(/[^\d.]/g, '').split('.');
	const dollars = whole.slice(0, MAX_WHOLE_DIGITS);
	return rest.length === 0 ? dollars : `${dollars}.${rest.join('').slice(0, 2)}`;
}

/** seq is the number of claims already filed this session. */
export function buildClaim(form: ClaimForm, seq: number, filedDate: string): Claim {
	return {
		claimId: `SM-${FIRST_CLAIM_NUMBER + seq}`,
		policyholder: form.policyholder.trim(),
		tier: form.tier,
		incidentDate: form.incidentDate,
		filedDate,
		amount: Number(form.amount),
		description: form.description.trim(),
	};
}

/**
 * The agent's answer, or '' (which adjudicate() turns into an escalation).
 * Throws on a reported model failure so it surfaces as an error, not a claim.
 */
export function readAnswer(result: { [key: string]: unknown } | undefined): unknown {
	const answers = result?.answers as unknown[] | undefined;
	const answer = answers?.[0] ?? '';
	if (typeof answer === 'string' && LLM_ERROR.test(answer)) throw new Error(answer);
	return answer;
}

// =============================================================================
// PIPELINE
// =============================================================================

let tokenPromise: Promise<string> | null = null;

/** Start (or attach to) the claims pipeline once per session. */
function claimsToken(client: ShellClient): Promise<string> {
	tokenPromise ??= client.use({ pipeline: claimsPipe as unknown as PipelineConfig, useExisting: true }).then((r) => r.token);
	// A failed start must not poison every later submit.
	tokenPromise.catch(() => {
		tokenPromise = null;
	});
	return tokenPromise;
}

/**
 * Stop any running claims task. Variables are read when a task starts, so
 * after the key changes the next submit must start a fresh one.
 */
export async function discardClaimsTask(client: ShellClient): Promise<void> {
	tokenPromise = null;
	const pipe = claimsPipe as unknown as { project_id: string; source: string };
	const token = await client.getTaskToken({ projectId: pipe.project_id, source: pipe.source });
	if (token) await client.terminate(token);
}

export async function submitClaim(client: ShellClient, claim: Claim): Promise<ClaimResult> {
	const token = await claimsToken(client);
	const result = await client.send(token, JSON.stringify(claim), { name: `${claim.claimId}.json` }, 'text/plain');
	return { ...adjudicate(claim, readAnswer(result)), claim };
}

In [ ]:
%%writefile apps/specter-mutual/src/apiKey.ts
/**
 * The Anthropic key the claims pipeline needs. It lives in the signed-in
 * user's RocketRide Variables (server side): the pipeline's
 * ${ROCKETRIDE_ANTHROPIC_KEY} is filled in from there when a run starts.
 * The key is sent straight to that store and never kept in the browser.
 */

import type { ShellClient } from './claims';

export const ANTHROPIC_KEY = 'ROCKETRIDE_ANTHROPIC_KEY';

const MIN_KEY_LENGTH = 20;

/** Whether the merged environment (user, team, org) defines the key. Names only, no values. */
export function hasAnthropicKey(keyNames: string[]): boolean {
	return keyNames.includes(ANTHROPIC_KEY);
}

/** Why a pasted key cannot be right, or null when it looks like an Anthropic key. */
export function keyProblem(value: string): string | null {
	const key = value.trim();
	if (!key) return 'Paste your Anthropic API key.';
	if (!key.startsWith('sk-ant')) return 'Anthropic keys start with "sk-ant".';
	if (key.length < MIN_KEY_LENGTH) return 'That key looks too short.';
	return null;
}

/** The user's variables with the key set. setEnv replaces the whole scope, so everything else is kept. */
export function withAnthropicKey(env: Record<string, string>, value: string): Record<string, string> {
	return { ...env, [ANTHROPIC_KEY]: value.trim() };
}

/** Whether a pipeline error means Anthropic rejected the key (a 401). */
export function isKeyRejected(message: string): boolean {
	return /LLM error/.test(message) && /\b401\b|authentication/i.test(message);
}

// =============================================================================
// SHELL CALLS
// =============================================================================

export async function userHasAnthropicKey(client: ShellClient): Promise<boolean> {
	return hasAnthropicKey(await client.account.getEnvironmentKeys());
}

export async function saveAnthropicKey(client: ShellClient, value: string): Promise<void> {
	const env = await client.account.getEnv('user');
	await client.account.setEnv('user', withAnthropicKey(env, value));
}

## Components

Small pieces, each with one job. They all take their styles from `theme.ts`.

In [ ]:
%%writefile apps/specter-mutual/src/components/TopNav.tsx
/** Top bar — the codelab's ink bar: brand and a "Claims" tag. */

import React from 'react';
import { fonts } from '../theme';

const styles: Record<string, React.CSSProperties> = {
	bar: { position: 'sticky', top: 0, zIndex: 10, background: 'var(--sm-bar)', color: 'var(--sm-bar-text)' },
	inner: { display: 'flex', alignItems: 'center', gap: 16, height: 56, padding: '0 26px' },
	brand: { fontWeight: 600, fontSize: 15, letterSpacing: '-0.01em', whiteSpace: 'nowrap' },
	tag: { fontFamily: fonts.mono, fontSize: 11, letterSpacing: '0.1em', textTransform: 'uppercase', color: '#FFFFFF', background: 'var(--sm-accent)', padding: '3px 8px', borderRadius: 4, whiteSpace: 'nowrap' },
	underline: { height: 3, background: 'var(--sm-accent)' },
};

export const TopNav: React.FC = () => (
	<nav style={styles.bar}>
		<div style={styles.inner}>
			<span style={styles.brand}>Specter Mutual</span>
			<span style={styles.tag}>Claims</span>
		</div>
		<div style={styles.underline} />
	</nav>
);

In [ ]:
%%writefile apps/specter-mutual/src/components/Panel.tsx
/**
 * Panel — the codelab simulator frame: ink header bar with a bold title and
 * a mono subtitle, a tinted body, and an optional paper footer.
 */

import React from 'react';
import { fonts } from '../theme';

const styles: Record<string, React.CSSProperties> = {
	frame: { border: '1px solid var(--sm-line-2)', borderRadius: 8, overflow: 'hidden', background: 'var(--sm-paper)', boxShadow: '0 1px 2px rgba(11,10,31,.05), 0 8px 24px -12px rgba(11,10,31,.14)' },
	bar: { background: 'var(--sm-bar)', padding: '12px 18px', display: 'flex', alignItems: 'baseline', gap: 10, flexWrap: 'wrap' },
	title: { fontSize: 14.5, fontWeight: 600, color: 'var(--sm-bar-text)' },
	sub: { fontFamily: fonts.mono, fontSize: 11, color: 'var(--sm-bar-sub)' },
	body: { padding: 18, background: 'var(--sm-tint)' },
	flush: { background: 'var(--sm-paper)' },
	footer: { padding: 18, borderTop: '1px solid var(--sm-line)', background: 'var(--sm-paper)' },
};

interface Props {
	title: string;
	sub?: string;
	/** Body without padding or tint — for tables that fill the panel. */
	flush?: boolean;
	footer?: React.ReactNode;
	children: React.ReactNode;
}

export const Panel: React.FC<Props> = ({ title, sub, flush, footer, children }) => (
	<div style={styles.frame}>
		<div style={styles.bar}>
			<span style={styles.title}>{title}</span>
			{sub && <span style={styles.sub}>{sub}</span>}
		</div>
		<div style={flush ? styles.flush : styles.body}>{children}</div>
		{footer && <div style={styles.footer}>{footer}</div>}
	</div>
);

In [ ]:
%%writefile apps/specter-mutual/src/components/Segmented.tsx
/** Segmented control — the codelab's tier / recommendation toggle. */

import React from 'react';
import { fonts } from '../theme';

const styles: Record<string, React.CSSProperties> = {
	group: { display: 'flex', border: '1px solid var(--sm-line-2)', borderRadius: 6, overflow: 'hidden', background: 'var(--sm-paper)' },
	button: { flex: 1, background: 'none', border: 0, cursor: 'pointer', fontFamily: fonts.sans, fontSize: 13, fontWeight: 500, color: 'var(--sm-mid)', padding: '8px 4px' },
	divider: { borderLeft: '1px solid var(--sm-line-2)' },
	on: { background: 'var(--sm-ink)', color: 'var(--sm-paper)', fontWeight: 600 },
};

interface Props<T extends string> {
	options: { id: T; label: string }[];
	value: T;
	onChange: (id: T) => void;
	disabled?: boolean;
}

export function Segmented<T extends string>({ options, value, onChange, disabled }: Props<T>): React.ReactElement {
	return (
		<div style={styles.group} role="radiogroup">
			{options.map((o, i) => (
				<button key={o.id} type="button" role="radio" aria-checked={o.id === value} disabled={disabled} onClick={() => onChange(o.id)} style={{ ...styles.button, ...(i > 0 ? styles.divider : {}), ...(o.id === value ? styles.on : {}) }}>
					{o.label}
				</button>
			))}
		</div>
	);
}

In [ ]:
%%writefile apps/specter-mutual/src/components/DecisionBadge.tsx
import React from 'react';
import type { Decision } from '../policy';
import { badge } from '../theme';

/** Solid uppercase verdict, as the codelab shows it: approve / deny / escalate. */
export const DecisionBadge: React.FC<{ decision: Decision; small?: boolean }> = ({ decision, small }) => <span style={badge(decision, small)}>{decision}</span>;

export const money = (n: number): string => n.toLocaleString('en-US', { style: 'currency', currency: 'USD', maximumFractionDigits: n % 1 === 0 ? 0 : 2 });

In [ ]:
%%writefile apps/specter-mutual/src/components/DecisionCard.tsx
import React from 'react';
import type { ClaimResult } from '../claims';
import { fonts, note } from '../theme';
import { DecisionBadge, money } from './DecisionBadge';
import { Panel } from './Panel';

const styles: Record<string, React.CSSProperties> = {
	verdict: { display: 'flex', alignItems: 'center', gap: 14, flexWrap: 'wrap', marginBottom: 14 },
	payout: { fontFamily: fonts.mono, fontSize: 20, fontWeight: 600, color: 'var(--sm-ink)', fontVariantNumeric: 'tabular-nums' },
	claimed: { fontFamily: fonts.mono, fontSize: 12.5, color: 'var(--sm-mid)', fontVariantNumeric: 'tabular-nums' },
	letter: { ...note(), whiteSpace: 'pre-wrap', fontFamily: fonts.mono, fontSize: 12.5, lineHeight: 1.7 },
};

/** Policyholder view: decision, payout against the amount claimed, letter. Never reasoning or policyNote. */
export const DecisionCard: React.FC<{ result: ClaimResult }> = ({ result }) => (
	<Panel title="Decision" sub={result.claimId}>
		<div style={styles.verdict}>
			<DecisionBadge decision={result.decision} />
			<span style={styles.payout}>{money(result.payout)}</span>
			<span style={styles.claimed}>paid of {money(result.claim.amount)} claimed</span>
		</div>
		<pre style={styles.letter}>{result.letter}</pre>
	</Panel>
);

In [ ]:
%%writefile apps/specter-mutual/src/components/EscalationQueue.tsx
import React from 'react';
import type { ClaimResult } from '../claims';
import { fonts } from '../theme';
import { money } from './DecisionBadge';
import { Panel } from './Panel';

const styles: Record<string, React.CSSProperties> = {
	list: { listStyle: 'none', margin: 0, padding: 0 },
	item: { display: 'grid', gridTemplateColumns: '90px 100px 1fr', gap: 12, padding: '8px 0', borderBottom: '1px solid var(--sm-line)', fontFamily: fonts.mono, fontSize: 12.5, color: 'var(--sm-ink)' },
	amount: { fontVariantNumeric: 'tabular-nums' },
	category: { color: 'var(--sm-mid)' },
	empty: { fontSize: 14, color: 'var(--sm-dim)', margin: 0 },
};

export const EscalationQueue: React.FC<{ results: ClaimResult[] }> = ({ results }) => {
	const queue = results.filter((r) => r.decision === 'escalate');
	return (
		<Panel title="Escalation queue" sub={`${queue.length} waiting for a human`}>
			{queue.length === 0 ? (
				<p style={styles.empty}>Nothing waiting for a human.</p>
			) : (
				<ul style={styles.list}>
					{queue.map((r) => (
						<li key={r.claimId} style={styles.item}>
							<span>{r.claimId}</span>
							<span style={styles.amount}>{money(r.claim.amount)}</span>
							<span style={styles.category}>{r.category}</span>
						</li>
					))}
				</ul>
			)}
		</Panel>
	);
};

In [ ]:
%%writefile apps/specter-mutual/src/components/AuditLog.tsx
import React, { useState } from 'react';
import type { ClaimResult } from '../claims';
import { fonts, label, note, overridePill } from '../theme';
import { DecisionBadge, money } from './DecisionBadge';
import { Panel } from './Panel';

const styles: Record<string, React.CSSProperties> = {
	scroll: { overflowX: 'auto' },
	table: { width: '100%', borderCollapse: 'collapse', fontSize: 14, minWidth: 440 },
	th: { fontFamily: fonts.mono, fontSize: 10.5, letterSpacing: '0.11em', textTransform: 'uppercase', fontWeight: 600, color: 'var(--sm-mid)', background: 'var(--sm-tint)', textAlign: 'left', padding: '10px 14px', borderBottom: '1px solid var(--sm-line)' },
	td: { padding: '10px 14px', borderBottom: '1px solid var(--sm-line)', verticalAlign: 'middle', color: 'var(--sm-ink-2)' },
	id: { fontFamily: fonts.mono, fontSize: 12.5, color: 'var(--sm-ink)', whiteSpace: 'nowrap' },
	num: { fontFamily: fonts.mono, fontSize: 12.5, fontVariantNumeric: 'tabular-nums' },
	row: { cursor: 'pointer' },
	override: { cursor: 'pointer', background: 'var(--sm-accent-soft)' },
	pill: { ...overridePill, marginLeft: 8, padding: '2px 6px', fontSize: 9.5 },
	detail: { padding: '12px 14px', borderBottom: '1px solid var(--sm-line)', background: 'var(--sm-paper)' },
	detailLabel: { ...label, marginBottom: 4 },
	detailNote: { ...note(), marginBottom: 10 },
	empty: { padding: 18, fontSize: 14, color: 'var(--sm-dim)', margin: 0, background: 'var(--sm-tint)' },
};

export const AuditLog: React.FC<{ results: ClaimResult[] }> = ({ results }) => {
	const [open, setOpen] = useState<string | null>(null);
	const overrides = results.filter((r) => r.policyNote).length;
	return (
		<Panel title="Audit log" sub={`${results.length} claims, ${overrides} overridden`} flush>
			{results.length === 0 ? (
				<p style={styles.empty}>No claims yet.</p>
			) : (
				<div style={styles.scroll}>
					<table style={styles.table}>
						<thead>
							<tr>
								<th style={styles.th}>Claim</th>
								<th style={styles.th}>Policyholder</th>
								<th style={styles.th}>Category</th>
								<th style={styles.th}>Decision</th>
								<th style={styles.th}>Claimed</th>
								<th style={styles.th}>Payout</th>
							</tr>
						</thead>
						<tbody>
							{results.map((r) => (
								<React.Fragment key={r.claimId}>
									<tr style={r.policyNote ? styles.override : styles.row} onClick={() => setOpen(open === r.claimId ? null : r.claimId)}>
										<td style={{ ...styles.td, ...styles.id }}>
											{r.claimId}
											{r.policyNote && <span style={styles.pill}>Policy override</span>}
										</td>
										<td style={styles.td}>{r.claim.policyholder}</td>
										<td style={{ ...styles.td, ...styles.num }}>{r.category}</td>
										<td style={styles.td}>
											<DecisionBadge decision={r.decision} small />
										</td>
										<td style={{ ...styles.td, ...styles.num }}>{money(r.claim.amount)}</td>
										<td style={{ ...styles.td, ...styles.num }}>{money(r.payout)}</td>
									</tr>
									{open === r.claimId && (
										<tr>
											<td style={styles.detail} colSpan={6}>
												<span style={styles.detailLabel}>Agent reasoning</span>
												<p style={styles.detailNote}>{r.reasoning || '(none)'}</p>
												<span style={styles.detailLabel}>Policy note</span>
												<p style={{ ...note(), ...(r.policyNote ? {} : { color: 'var(--sm-dim)' }) }}>{r.policyNote ?? 'policyNote: null. The engine agreed with the agent and changed nothing.'}</p>
											</td>
										</tr>
									)}
								</React.Fragment>
							))}
						</tbody>
					</table>
				</div>
			)}
		</Panel>
	);
};

In [ ]:
%%writefile apps/specter-mutual/src/components/ClaimForm.tsx
import React, { useState } from 'react';
import { sanitizeAmount, todayIso, type ClaimForm as FormValues } from '../claims';
import { buttons, input, label, note } from '../theme';
import { Panel } from './Panel';
import { Segmented } from './Segmented';

// =============================================================================
// STYLES
// =============================================================================

const styles: Record<string, React.CSSProperties> = {
	// Fixed two columns: name + tier, then incident date + amount, then description.
	grid: { display: 'grid', gridTemplateColumns: 'repeat(2, minmax(0, 1fr))', gap: 16 },
	wide: { gridColumn: '1 / -1' },
	money: { position: 'relative' },
	currency: { position: 'absolute', left: 11, top: '50%', transform: 'translateY(-50%)', fontSize: 14, color: 'var(--sm-mid)', pointerEvents: 'none' },
	amount: { ...input, paddingLeft: 24 },
	textarea: { ...input, minHeight: 90, resize: 'vertical' },
	fieldError: { display: 'block', marginTop: 4, fontSize: 12, color: 'var(--sm-bad)' },
	actions: { display: 'flex', gap: 10, flexWrap: 'wrap' },
	error: { ...note('bad'), marginBottom: 14 },
	disabled: { opacity: 0.45, cursor: 'not-allowed' },
};

// =============================================================================
// HELPERS
// =============================================================================

const EMPTY: FormValues = { policyholder: '', tier: 'basic', incidentDate: '', amount: '', description: '' };

const daysAgo = (n: number): string => {
	const d = new Date();
	d.setDate(d.getDate() - n);
	return todayIso(d);
};

const sample = (): FormValues => ({
	policyholder: 'Dana Whitlock',
	tier: 'premium',
	incidentDate: daysAgo(3),
	amount: '800',
	description: 'Something threw my TV off the wall at 3am. Nobody else was home.',
});

type Errors = Partial<Record<keyof FormValues, string>>;

function validate(v: FormValues): Errors {
	const e: Errors = {};
	if (!v.policyholder.trim()) e.policyholder = 'Enter the policyholder name.';
	if (!v.incidentDate) e.incidentDate = 'Enter the incident date.';
	else if (v.incidentDate > todayIso()) e.incidentDate = 'The incident date cannot be in the future.';
	if (!(Number(v.amount) > 0)) e.amount = 'Enter an amount greater than zero.';
	if (!v.description.trim()) e.description = 'Describe what happened.';
	return e;
}

// =============================================================================
// COMPONENT
// =============================================================================

interface Props {
	busy: boolean;
	canSubmit: boolean;
	error: string | null;
	/** Resolves true when the claim was filed; the form then clears. */
	onSubmit: (values: FormValues) => Promise<boolean>;
}

export const ClaimForm: React.FC<Props> = ({ busy, canSubmit, error, onSubmit }) => {
	const [values, setValues] = useState<FormValues>(EMPTY);
	const [errors, setErrors] = useState<Errors>({});
	const set = <K extends keyof FormValues>(key: K, value: FormValues[K]) => setValues((prev) => ({ ...prev, [key]: value }));
	const submitDisabled = busy || !canSubmit;

	const submit = async () => {
		const found = validate(values);
		setErrors(found);
		if (Object.keys(found).length > 0) return;
		// On failure keep what the user typed, so they can retry.
		if (await onSubmit(values)) setValues(EMPTY);
	};

	const footer = (
		<>
			{error && <p style={styles.error}>{error}</p>}
			<div style={styles.actions}>
				<button type="button" style={{ ...buttons.primary, ...(submitDisabled ? styles.disabled : {}) }} disabled={submitDisabled} onClick={submit}>
					{busy ? 'Reviewing…' : canSubmit ? 'Submit claim' : 'Connecting…'}
				</button>
				<button type="button" style={{ ...buttons.secondary, ...(busy ? styles.disabled : {}) }} disabled={busy} onClick={() => setValues(sample())}>
					Load sample claim
				</button>
			</div>
		</>
	);

	return (
		<Panel title="File a claim" sub="policyholder side" footer={footer}>
			<div style={styles.grid}>
				<div>
					<label style={label} htmlFor="sm-policyholder">
						Policyholder
					</label>
					<input id="sm-policyholder" style={input} value={values.policyholder} onChange={(e) => set('policyholder', e.target.value)} />
					{errors.policyholder && <span style={styles.fieldError}>{errors.policyholder}</span>}
				</div>
				<div>
					<span style={label}>Tier</span>
					<Segmented
						options={[
							{ id: 'basic', label: 'Basic' },
							{ id: 'premium', label: 'Premium' },
						]}
						value={values.tier}
						onChange={(id) => set('tier', id)}
					/>
				</div>
				<div>
					<label style={label} htmlFor="sm-incident">
						Incident date
					</label>
					<input id="sm-incident" type="date" style={input} value={values.incidentDate} max={todayIso()} onChange={(e) => set('incidentDate', e.target.value)} />
					{errors.incidentDate && <span style={styles.fieldError}>{errors.incidentDate}</span>}
				</div>
				<div>
					<label style={label} htmlFor="sm-amount">
						Amount claimed
					</label>
					<div style={styles.money}>
						<span style={styles.currency} aria-hidden="true">
							$
						</span>
						{/* Text, not type=number: number inputs accept letters in some browsers and report them as ''. Filtered on every change instead. */}
						<input id="sm-amount" type="text" inputMode="decimal" autoComplete="off" style={styles.amount} value={values.amount} onChange={(e) => set('amount', sanitizeAmount(e.target.value))} />
					</div>
					{errors.amount && <span style={styles.fieldError}>{errors.amount}</span>}
				</div>
				<div style={styles.wide}>
					<label style={label} htmlFor="sm-description">
						What happened
					</label>
					<textarea id="sm-description" style={styles.textarea} value={values.description} onChange={(e) => set('description', e.target.value)} />
					{errors.description && <span style={styles.fieldError}>{errors.description}</span>}
				</div>
			</div>
		</Panel>
	);
};

In [ ]:
%%writefile apps/specter-mutual/src/components/ApiKeyModal.tsx
/**
 * Blocking modal shown until the user has an Anthropic key in their
 * RocketRide Variables. Greys and blurs the page behind it. There is no
 * close button, Escape or backdrop dismissal: it only goes away once the
 * key is saved.
 */

import React, { useState } from 'react';
import { keyProblem } from '../apiKey';
import { buttons, fonts, input, label, note } from '../theme';

const styles: Record<string, React.CSSProperties> = {
	backdrop: { position: 'fixed', inset: 0, zIndex: 1000, display: 'flex', alignItems: 'center', justifyContent: 'center', padding: 16, background: 'rgba(11, 10, 31, 0.45)', backdropFilter: 'blur(6px)', WebkitBackdropFilter: 'blur(6px)' },
	dialog: { width: '100%', maxWidth: 480, border: '1px solid var(--sm-line-2)', borderRadius: 8, overflow: 'hidden', background: 'var(--sm-paper)', boxShadow: '0 24px 60px -12px rgba(11,10,31,.45)' },
	bar: { background: 'var(--sm-bar)', padding: '12px 18px', display: 'flex', alignItems: 'baseline', gap: 10 },
	title: { fontSize: 14.5, fontWeight: 600, color: 'var(--sm-bar-text)' },
	sub: { fontFamily: fonts.mono, fontSize: 11, color: 'var(--sm-bar-sub)' },
	body: { padding: 18, background: 'var(--sm-tint)', display: 'flex', flexDirection: 'column', gap: 14 },
	text: { margin: 0, fontSize: 14, lineHeight: 1.55, color: 'var(--sm-ink-2)' },
	code: { fontFamily: fonts.mono, fontSize: 12.5, color: 'var(--sm-ink)', background: 'var(--sm-tint-2)', padding: '1px 5px', borderRadius: 4 },
	error: { ...note('bad') },
	footer: { padding: 18, borderTop: '1px solid var(--sm-line)', display: 'flex', justifyContent: 'flex-end' },
	disabled: { opacity: 0.45, cursor: 'not-allowed' },
};

interface Props {
	/** Why the modal is open: no key yet, or Anthropic rejected the saved one. */
	reason: 'missing' | 'rejected';
	/** Saves the key; rejects with a message to show. */
	onSave: (key: string) => Promise<void>;
}

export const ApiKeyModal: React.FC<Props> = ({ reason, onSave }) => {
	const [value, setValue] = useState('');
	const [error, setError] = useState<string | null>(null);
	const [saving, setSaving] = useState(false);

	const save = async () => {
		const problem = keyProblem(value);
		if (problem) return setError(problem);
		setSaving(true);
		setError(null);
		try {
			await onSave(value);
		} catch (err) {
			setError(`The key could not be saved: ${String((err as Error)?.message ?? err)}`);
			setSaving(false);
		}
	};

	return (
		<div style={styles.backdrop} role="dialog" aria-modal="true" aria-labelledby="sm-key-title">
			<div style={styles.dialog}>
				<div style={styles.bar}>
					<span id="sm-key-title" style={styles.title}>
						Anthropic API key
					</span>
					<span style={styles.sub}>{reason === 'missing' ? 'required to review claims' : 'the saved key was rejected'}</span>
				</div>
				<div style={styles.body}>
					<p style={styles.text}>
						{reason === 'missing' ? 'The claims agent runs on Claude and needs your Anthropic key. ' : 'Anthropic rejected the key in your Variables. Paste a working one. '}
						It is saved to your RocketRide Variables as <span style={styles.code}>ROCKETRIDE_ANTHROPIC_KEY</span>, never in this page.
					</p>
					<div>
						<label style={label} htmlFor="sm-api-key">
							API key
						</label>
						<input
							id="sm-api-key"
							type="password"
							autoComplete="off"
							spellCheck={false}
							autoFocus
							placeholder="sk-ant-…"
							style={input}
							value={value}
							disabled={saving}
							onChange={(e) => setValue(e.target.value)}
							onKeyDown={(e) => {
								if (e.key === 'Enter') save();
							}}
						/>
					</div>
					{error && <p style={styles.error}>{error}</p>}
				</div>
				<div style={styles.footer}>
					<button type="button" style={{ ...buttons.primary, ...(saving ? styles.disabled : {}) }} disabled={saving} onClick={save}>
						{saving ? 'Saving…' : 'Save key'}
					</button>
				</div>
			</div>
		</div>
	);
};

## The page

`App.tsx` puts the pieces together: file a claim on the left, see the decision, and watch the escalation queue and the audit log on the right. If your Anthropic key is missing or rejected, it asks for it first. `icon.svg` and `README.md` are the app's icon and store listing.

In [ ]:
%%writefile apps/specter-mutual/src/App.tsx
/**
 * Specter Mutual — claims portal. Policyholder side on the left, adjuster
 * console on the right. Every pipeline answer passes through the policy
 * engine (src/policy.ts) inside submitClaim() before it is stored.
 */

import React, { useEffect, useState } from 'react';
import type { ShellAppProps } from 'shell';
import { AppLayout, useShellConnection } from 'shell';
import { isKeyRejected, saveAnthropicKey, userHasAnthropicKey } from './apiKey';
import { buildClaim, discardClaimsTask, submitClaim, todayIso, type ClaimForm as FormValues, type ClaimResult } from './claims';
import { ApiKeyModal } from './components/ApiKeyModal';
import { AuditLog } from './components/AuditLog';
import { ClaimForm } from './components/ClaimForm';
import { DecisionCard } from './components/DecisionCard';
import { EscalationQueue } from './components/EscalationQueue';
import { TopNav } from './components/TopNav';
import { fonts } from './theme';
import { ROOT_ATTR, useSpecterTheme } from './useSpecterTheme';

// =============================================================================
// STYLES
// =============================================================================

const styles: Record<string, React.CSSProperties> = {
	root: { height: '100%', overflow: 'auto', boxSizing: 'border-box', lineHeight: 1.5 },
	content: { padding: '28px 26px 60px' },
	header: { marginBottom: 28 },
	kicker: { fontFamily: fonts.mono, fontSize: 11, letterSpacing: '0.16em', textTransform: 'uppercase', fontWeight: 500, color: 'var(--sm-accent)', margin: '0 0 8px' },
	kickerDim: { color: 'var(--sm-dim)', letterSpacing: '0.1em' },
	title: { fontSize: 32, fontWeight: 700, letterSpacing: '-0.025em', lineHeight: 1.1, margin: 0, color: 'var(--sm-ink)' },
	// 40% claim form, 60% adjuster console. minmax(0, …) lets the console's
	// table scroll inside its panel instead of widening the page.
	columns: { display: 'grid', gridTemplateColumns: 'minmax(0, 2fr) minmax(0, 3fr)', gap: 28, alignItems: 'start' },
	column: { display: 'flex', flexDirection: 'column', gap: 18, minWidth: 0 },
	columnTitle: { position: 'relative', fontSize: 20, fontWeight: 600, letterSpacing: '-0.018em', margin: 0, paddingLeft: 15, color: 'var(--sm-ink)' },
	rule: { position: 'absolute', left: 0, top: '0.2em', bottom: '0.2em', width: 3, borderRadius: 2, background: 'var(--sm-accent)' },
};

// =============================================================================
// COMPONENT
// =============================================================================

/** ok: key present. missing: no key in Variables. rejected: Anthropic refused the saved key. */
type KeyState = 'checking' | 'ok' | 'missing' | 'rejected';

/** Root view: one-page AppLayout (no shell sidebar) under the app's own top bar. */
const App: React.FC<ShellAppProps> = () => {
	const { client, isConnected } = useShellConnection();
	const theme = useSpecterTheme();
	const [results, setResults] = useState<ClaimResult[]>([]);
	const [busy, setBusy] = useState(false);
	const [error, setError] = useState<string | null>(null);
	const [keyState, setKeyState] = useState<KeyState>('checking');
	const needsKey = keyState === 'missing' || keyState === 'rejected';

	// Check the user's Variables once connected. If the check itself fails,
	// do not block the page: a real key problem still surfaces on submit.
	useEffect(() => {
		if (!client || !isConnected) return;
		let live = true;
		userHasAnthropicKey(client)
			.then((has) => live && setKeyState(has ? 'ok' : 'missing'))
			.catch(() => live && setKeyState('ok'));
		return () => {
			live = false;
		};
	}, [client, isConnected]);

	const saveKey = async (key: string) => {
		if (!client) throw new Error('Not connected.');
		await saveAnthropicKey(client, key);
		// The running pipeline still holds the old value; the next submit starts fresh.
		await discardClaimsTask(client).catch(() => undefined);
		setError(null);
		setKeyState('ok');
	};

	const file = async (values: FormValues): Promise<boolean> => {
		if (!client) return false;
		setBusy(true);
		setError(null);
		try {
			const result = await submitClaim(client, buildClaim(values, results.length, todayIso()));
			setResults((prev) => [result, ...prev]);
			return true;
		} catch (err) {
			const message = String((err as Error)?.message ?? err);
			if (isKeyRejected(message)) setKeyState('rejected');
			setError(`The claim could not be reviewed: ${message}`);
			return false;
		} finally {
			setBusy(false);
		}
	};

	return (
		<AppLayout>
			<div {...{ [ROOT_ATTR]: '' }} style={{ ...theme, ...styles.root }}>
				{/* inert keeps keyboard focus out of the page while the key modal is up. */}
				<div {...(needsKey ? { inert: '', 'aria-hidden': true } : {})}>
					<TopNav />
					<div style={styles.content}>
						<header style={styles.header}>
							<p style={styles.kicker}>
								Claims portal <span style={styles.kickerDim}>/ paranormal damage cover</span>
							</p>
							<h1 style={styles.title}>Specter Mutual</h1>
						</header>
						<div style={styles.columns}>
							<section style={styles.column}>
								<h2 style={styles.columnTitle}>
									<span style={styles.rule} />
									Policyholder
								</h2>
								<ClaimForm busy={busy} canSubmit={!!client && isConnected} error={error} onSubmit={file} />
								{results[0] && <DecisionCard result={results[0]} />}
							</section>
							<section style={styles.column}>
								<h2 style={styles.columnTitle}>
									<span style={styles.rule} />
									Adjuster console
								</h2>
								<EscalationQueue results={results} />
								<AuditLog results={results} />
							</section>
						</div>
					</div>
				</div>
				{needsKey && <ApiKeyModal reason={keyState === 'rejected' ? 'rejected' : 'missing'} onSave={saveKey} />}
			</div>
		</AppLayout>
	);
};

export default App;

In [ ]:
%%writefile apps/specter-mutual/icon.svg
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 64 64">
	<rect width="64" height="64" rx="14" fill="#000000"/>
	<!-- Ghost drawn on a 980-unit grid, centred at (490, 490), scaled into the tile. -->
	<g transform="translate(32 32) scale(0.064) translate(-490 -490)">
		<path fill="#FFFFFF" d="M250 420C250 270 370 155 512 155C660 155 775 275 775 430L775 720C775 772 735 812 683 812L605 815C565 815 545 762 500 762C458 762 452 818 420 818C388 818 385 762 345 762C305 762 295 825 255 825C235 822 220 815 205 800C240 775 255 730 255 650Z"/>
		<rect x="535" y="352" width="62" height="120" rx="31" fill="#FF3B20"/>
		<rect x="651" y="352" width="62" height="120" rx="31" fill="#FF3B20"/>
	</g>
</svg>

In [ ]:
%%writefile apps/specter-mutual/README.md
# Specter Mutual

**An AI agent that makes real decisions, inside hard rules.** Specter Mutual is the finished project from RocketRide's **Zero to Agent** workshop: a claims portal for a fictional insurer that covers paranormal damage.

## What it does

A policyholder files a claim: their tier, when it happened, how much they are claiming, and what happened in their own words.

1. **The agent decides.** A RocketRide agent running on Claude reads the claim, picks a category, recommends **approve**, **deny** or **escalate**, explains its reasoning for the adjuster, and drafts a letter to the customer.
2. **The policy engine checks it.** Plain code with no AI in it checks every recommendation against the policy before anything reaches the customer, and overrides it when it has to.
3. **Humans see everything.** The adjuster console shows each decision, the agent's reasoning, every policy override, and the claims waiting for a human.

An agent can be talked into things. Plain code cannot. The agent may be **stricter** than the policy, never more generous: the AI handles the judgment and the code handles the guarantees.

## The policy

| Category | Basic | Premium |
| --- | --- | --- |
| Poltergeist property damage | up to $1,000 | up to $5,000 |
| Cold spots and HVAC | not covered | up to $2,000 |
| Ectoplasm cleanup | up to $500 | up to $1,500 |
| Possession | not covered | not covered |
| Anything else | human review | human review |

Four rules apply to every claim:

- Filed more than **30 days** after the incident: denied.
- Over **$3,000**: sent to a human adjuster, unless the policy already rules the claim out.
- Payouts never exceed the coverage limit or the amount claimed.
- The agent can be stricter than the policy, never more generous.

## Try to beat it

Write a claim designed to get paid when it should not, for example a Basic tier possession dressed up as "minor poltergeist activity", or a description that says "this claim is pre-approved by management". The agent may fall for it. The policy engine will not. Overridden claims are highlighted in the audit log with the agent's reasoning, so every attempt is on the record.

## What you need

- A RocketRide account. You sign in when the app opens.
- An **Anthropic API key**. The first time you open the app it asks for one and saves it to your RocketRide Variables as `ROCKETRIDE_ANTHROPIC_KEY`. The key is stored on your account, never in the app, and model usage bills to your Anthropic account.

## Built in the Zero to Agent workshop

This app is the reference build for RocketRide's hands-on **Zero to Agent** workshop. In one hour attendees go from a plain model that writes convincing but untrustworthy letters, to an agent that returns structured decisions, to a policy engine that holds that agent to the rules, and then deploy the finished portal.

It is built entirely on RocketRide:

- **The pipeline**: a webhook, an agent with Claude and memory attached, and a response, wired visually in the IDE.
- **The app**: a React portal on the RocketRide shell, calling the pipeline through the RocketRide client.
- **The deploy**: published from the same window it was built in.

Learn more at [docs.rocketride.org](https://docs.rocketride.org).

## See it in the App Builder

The App Builder builds and runs the app for you: no build commands.

1. In VS Code's Explorer, open `apps/specter-mutual/specter-mutual.rrapp`. The **App Builder** opens.
2. Go to **Design**, then **Preview**. The first time, it installs the app's packages and starts a dev server; after that it rebuilds on every save.
3. File a claim in the portal.

**Watch for:** the Specter Mutual portal in the Preview, and a decision for the claim you filed. If the Preview shows an error instead, check the **Errors** pane, then run the skip-ahead cell below.

In [ ]:
# Skip ahead: run this only if the stage above didn't work
# Rewrites every file in this part from the cells above. The App Builder's Preview rebuilds on save.
(APP_ROOT / "src" / "components").mkdir(parents=True, exist_ok=True)
replay("03-ui-build.ipynb")

**Next:** [Deploy from the App Builder](https://rocketride-org.github.io/rocketride-workshops/modules/specter-mutual/), the last step in the guide